# Ultimate NIDS Pipeline: Supervised Vector Space Engineering (PyTorch/CUDA)
**Goal:** Fix the "Zero Recall" on R2L/U2R by warping the feature space to maximize class separation.

**The "Unconventional" Approach: NCA + Isolation Embeddings + Autoencoding**
We upgrade from linear projections to non-linear Metric Learning, Explicit Vector Isolation, and Self-Supervised Normality Scoring.

**New Architecture:**
1.  **Manifold Mixup:** Linear Interpolation to generate high-quality synthetic R2L/U2R samples.
2.  **Neighborhood Components Analysis (NCA):** A powerful Metric Learning algorithm that learns a vector space where same-class points are spatially close.
3.  **Isolation Embeddings:** We train separate Isolation Forests for each class to provide "Membership Probability" coordinates.
4.  **Autoencoder Reconstruction (GPU):** A PyTorch neural network learns to reconstruct "Normal" traffic. The reconstruction error serves as a powerful "Weirdness Score".
5.  **Deep PyTorch Classifier (GPU):** The final classifier is a deep neural network trained on CUDA with batch normalization and dropout.

## 1. Data Loading

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.neighbors import NeighborhoodComponentsAnalysis
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, LabelEncoder, QuantileTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, matthews_corrcoef, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import math
import warnings
import gc

# Config
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 PIPELINE: FT-TRANSFORMER + TRIPLET LOSS on: {device}")

# ==========================================
# 1. SETUP & DATA LOADING (Same as before)
# ==========================================
attack_map = {
    'normal': 'normal',
    'neptune': 'dos', 'back': 'dos', 'land': 'dos', 'pod': 'dos', 'smurf': 'dos', 'teardrop': 'dos', 'mailbomb': 'dos', 'apache2': 'dos', 'processtable': 'dos', 'udpstorm': 'dos', 
    'ipsweep': 'probe', 'nmap': 'probe', 'portsweep': 'probe', 'satan': 'probe', 'mscan': 'probe', 'saint': 'probe',
    'ftp_write': 'r2l', 'guess_passwd': 'r2l', 'imap': 'r2l', 'multihop': 'r2l', 'phf': 'r2l', 'spy': 'r2l', 'warezclient': 'r2l', 'warezmaster': 'r2l', 'sendmail': 'r2l', 'named': 'r2l', 'snmpgetattack': 'r2l', 'snmpguess': 'r2l', 'xlock': 'r2l', 'xsnoop': 'r2l', 'worm': 'r2l', 'httptunnel': 'r2l',
    'buffer_overflow': 'u2r', 'loadmodule': 'u2r', 'perl': 'u2r', 'rootkit': 'u2r', 'ps': 'u2r', 'sqlattack': 'u2r', 'xterm': 'u2r'
}

def load_and_prep(path):
    print(f"Loading {path}...")
    df = pd.read_csv(path)
    df['label'] = df['label'].astype(str).str.replace('.', '', regex=False)
    df['category'] = df['label'].map(attack_map).fillna('other')
    for c in ['src_bytes', 'dst_bytes', 'duration', 'wrong_fragment', 'urgent']:
        if c in df.columns: df[c] = np.log1p(df[c])
    X = df.drop(['label', 'category'], axis=1).select_dtypes(include=[np.number])
    y = df['category']
    return X, y

X, y = load_and_prep('Data/network_connections.csv')
le = LabelEncoder()
y_vec = le.fit_transform(y)
num_classes = len(le.classes_)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y_vec, test_size=0.2, stratify=y_vec, random_state=42)
train_cols = X.columns.tolist()

del X, y; gc.collect()

# ==========================================
# 2. FEATURE ENGINEERING
# ==========================================
print("\n--- PHASE A: ENGINEERING RICH FEATURES ---")
scaler = QuantileTransformer(output_distribution='normal', random_state=42) 
X_train_sc = scaler.fit_transform(X_train_raw).astype(np.float32)

print("1. NCA...")
nca = NeighborhoodComponentsAnalysis(n_components=15, random_state=42)
idx_sample = np.random.choice(len(X_train_sc), size=min(5000, len(X_train_sc)), replace=False)
nca.fit(X_train_sc[idx_sample], y_train[idx_sample])
X_train_nca = nca.transform(X_train_sc).astype(np.float32)

print("2. Isolation Forests...")
iso_models = {}
iso_feats_train = []
for cls_idx in np.unique(y_train):
    X_cls = X_train_sc[y_train == cls_idx]
    if len(X_cls) < 50: continue
    iso = IsolationForest(n_estimators=50, contamination=0.01, n_jobs=-1, random_state=42)
    iso.fit(X_cls)
    iso_models[cls_idx] = iso
    iso_feats_train.append(iso.decision_function(X_train_sc).reshape(-1, 1).astype(np.float32))

X_train_iso = np.hstack(iso_feats_train)

print("3. Autoencoder...")
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, 32), nn.ReLU(), nn.Linear(32, input_dim))
    def forward(self, x): return self.net(x)

ae = Autoencoder(X_train_sc.shape[1]).to(device)
ae_opt = optim.Adam(ae.parameters(), lr=0.005)
ae_loader = DataLoader(TensorDataset(torch.tensor(X_train_sc)), batch_size=512, shuffle=True)
for _ in range(10): # Shortened for brevity
    for batch in ae_loader:
        x = batch[0].to(device)
        loss = F.mse_loss(ae(x), x)
        ae_opt.zero_grad(); loss.backward(); ae_opt.step()

def get_recon(data):
    ae.eval()
    loader = DataLoader(TensorDataset(torch.tensor(data)), batch_size=2048)
    errs = []
    with torch.no_grad():
        for b in loader:
            x = b[0].to(device)
            errs.append(torch.mean((x - ae(x))**2, dim=1).cpu().numpy())
    return np.concatenate(errs).reshape(-1,1).astype(np.float32)

X_train_ae = get_recon(X_train_sc)
X_train_final = np.hstack([X_train_sc, X_train_nca, X_train_iso, X_train_ae])
X_train_tensor = torch.tensor(X_train_final, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

del X_train_sc, X_train_nca, X_train_iso, X_train_ae; gc.collect()

# ==========================================
# 3. LOSS FUNCTION: BATCH HARD TRIPLET
# ==========================================
class BatchHardTripletLoss(nn.Module):
    """
    Calculates the triplet loss in a batch in a smart way.
    For every anchor, we pick the HARDEST positive (furthest away) 
    and the HARDEST negative (closest).
    """
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, embeddings, labels):
        # Pairwise distances: [B, B]
        dist_mat = torch.cdist(embeddings, embeddings)
        
        # Positive Mask: same label, not self
        is_pos = labels.unsqueeze(0) == labels.unsqueeze(1)
        is_pos.fill_diagonal_(False)
        
        # Negative Mask: different label
        is_neg = labels.unsqueeze(0) != labels.unsqueeze(1)
        
        # Hardest Positive: Max distance among positives
        dist_pos = dist_mat.clone()
        dist_pos[~is_pos] = -float('inf')
        hard_pos, _ = dist_pos.max(dim=1)
        
        # Hardest Negative: Min distance among negatives
        dist_neg = dist_mat.clone()
        dist_neg[~is_neg] = float('inf')
        hard_neg, _ = dist_neg.min(dim=1)
        
        # Loss = ReLU(Hard_Pos - Hard_Neg + Margin)
        loss = F.relu(hard_pos - hard_neg + self.margin)
        
        return loss.mean()

# ==========================================
# 4. FT-TRANSFORMER (Modified for Embeddings)
# ==========================================
class FeatureTokenizer(nn.Module):
    def __init__(self, num_features, embed_dim):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(num_features, embed_dim) / np.sqrt(embed_dim))
        self.biases = nn.Parameter(torch.zeros(num_features, embed_dim))
    def forward(self, x):
        return x.unsqueeze(-1) * self.weights.unsqueeze(0) + self.biases.unsqueeze(0)

class FTTransformerTriplet(nn.Module):
    def __init__(self, num_features, num_classes, embed_dim=128, depth=3, heads=4, dropout=0.1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(num_features, embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self._init_weights()
        
        enc = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=heads, dim_feedforward=embed_dim*2, 
                                       dropout=dropout, batch_first=True, norm_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(enc, num_layers=depth)
        
        self.norm = nn.LayerNorm(embed_dim)
        
        # Projection head for Triplet (Optional, but good practice)
        self.embed_head = nn.Linear(embed_dim, embed_dim) 
        
        # Classification Head
        self.cls_head = nn.Linear(embed_dim, num_classes)
        
    def _init_weights(self):
        nn.init.normal_(self.cls_token, std=0.02)

    def forward(self, x):
        x_emb = self.tokenizer(x)
        b = x.shape[0]
        x_seq = torch.cat((self.cls_token.expand(b, -1, -1), x_emb), dim=1)
        x_out = self.transformer(x_seq)
        
        # Latent representation (CLS token)
        latent = self.norm(x_out[:, 0, :])
        
        # 1. Embedding for Triplet Loss (Normalized)
        embedding = F.normalize(self.embed_head(latent), p=2, dim=1)
        
        # 2. Logits for CE Loss
        logits = self.cls_head(latent)
        
        return logits, embedding

# ==========================================
# 5. TRAINING LOOP
# ==========================================
BATCH_SIZE = 256
EPOCHS = 20

model = FTTransformerTriplet(X_train_final.shape[1], num_classes).to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)

# Loaders
train_dl = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True)
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.001, steps_per_epoch=len(train_dl), epochs=EPOCHS)

# Losses
criterion_ce = nn.CrossEntropyLoss(label_smoothing=0.1)
criterion_triplet = BatchHardTripletLoss(margin=0.5)

print("Training Transformer with Dual Loss...")
for epoch in range(EPOCHS):
    model.train()
    loss_acc = 0
    tri_acc = 0
    
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        
        # Forward gives both
        logits, embedding = model(x)
        
        # Calculate individual losses
        l_ce = criterion_ce(logits, y)
        l_tri = criterion_triplet(embedding, y)
        
        # Combine: Usually weighted. Triplet needs time to stabilize.
        loss = l_ce + (1.0 * l_tri)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        loss_acc += loss.item()
        tri_acc += l_tri.item()
        
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1:02d} | Total Loss: {loss_acc/len(train_dl):.4f} | Triplet Part: {tri_acc/len(train_dl):.4f}")

# ==========================================
# 6. EVALUATION
# ==========================================
print("\n--- PHASE E: TUNING INFERENCE ---")
try:
    NSL_URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.txt"
    NSL_COLS = ['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty_level']
    df_nsl = pd.read_csv(NSL_URL, header=None, names=NSL_COLS)
    if 'difficulty_level' in df_nsl.columns: df_nsl.drop('difficulty_level', axis=1, inplace=True)
    df_nsl['label'] = df_nsl['label'].astype(str).str.replace('.', '', regex=False)
    df_nsl['category'] = df_nsl['label'].map(attack_map).fillna('other')
    for c in ['src_bytes', 'dst_bytes', 'duration', 'wrong_fragment', 'urgent']:
        if c in df_nsl.columns: df_nsl[c] = np.log1p(df_nsl[c])
            
    df_nsl_aligned = pd.DataFrame(0, index=np.arange(len(df_nsl)), columns=train_cols)
    for c in train_cols:
        if c in df_nsl.columns: df_nsl_aligned[c] = df_nsl[c]
    y_nsl_vec = df_nsl['category'].apply(lambda x: le.transform([x])[0] if x in le.classes_ else -1).values
    
    # Transform
    X_nsl_sc = scaler.transform(df_nsl_aligned).astype(np.float32)
    X_nsl_nca = nca.transform(X_nsl_sc).astype(np.float32)
    iso_feats_nsl = []
    for cls_idx in np.sort(list(iso_models.keys())):
        iso_feats_nsl.append(iso_models[cls_idx].decision_function(X_nsl_sc).reshape(-1, 1).astype(np.float32))
    X_nsl_iso = np.hstack(iso_feats_nsl)
    X_nsl_ae = get_recon(X_nsl_sc)
    
    X_nsl_final = np.hstack([X_nsl_sc, X_nsl_nca, X_nsl_iso, X_nsl_ae])
    X_test_tensor_cpu = torch.tensor(X_nsl_final)

    def predict_batched(model, x_cpu):
        model.eval()
        x_gpu = x_cpu.to(device)
        with torch.no_grad():
            # Model returns (logits, embedding), we only want logits [0]
            logits, _ = model(x_gpu)
            probs = torch.softmax(logits, dim=1)
        return probs.cpu().numpy()

    all_probs = []
    loader = DataLoader(TensorDataset(X_test_tensor_cpu), batch_size=2048)
    for bx in loader:
        p = predict_batched(model, bx[0]) 
        all_probs.append(p)
    probs_np = np.concatenate(all_probs)
    
    # Thresholding
    idx_u2r = list(le.classes_).index('u2r')
    idx_r2l = list(le.classes_).index('r2l')
    best_score = -1; best_params = (0, 0)
    
    print("Searching best thresholds...")
    for t1 in [0.005, 0.01, 0.05, 0.1]:
        for t2 in [0.005, 0.01, 0.05, 0.1]:
            tp = []
            for p in probs_np:
                if p[idx_u2r] > t2: tp.append(idx_u2r)
                elif p[idx_r2l] > t1: tp.append(idx_r2l)
                else: tp.append(np.argmax(p))
            mask = y_nsl_vec != -1
            mcc = matthews_corrcoef(y_nsl_vec[mask], np.array(tp)[mask])
            if mcc > best_score: best_score = mcc; best_params = (t1, t2)
    
    final_preds = []
    for p in probs_np:
        if p[idx_u2r] > best_params[1]: final_preds.append(idx_u2r)
        elif p[idx_r2l] > best_params[0]: final_preds.append(idx_r2l)
        else: final_preds.append(np.argmax(p))
    final_preds = np.array(final_preds)

    mask = y_nsl_vec != -1
    acc = accuracy_score(y_nsl_vec[mask], final_preds[mask])
    mcc = matthews_corrcoef(y_nsl_vec[mask], final_preds[mask])
    
    print("\n" + "="*40)
    print(f"   👑 RESULTS (FT-Transformer + Triplet): {acc:.2%} (MCC: {mcc:.4f}) 👑")
    print("="*40)
    print(classification_report(y_nsl_vec[mask], final_preds[mask], target_names=le.classes_))
    
except Exception as e:
    print(f"Error: {e}")
    import traceback; traceback.print_exc()

🚀 PIPELINE: FT-TRANSFORMER + TRIPLET LOSS on: cuda
Loading Data/network_connections.csv...

--- PHASE A: ENGINEERING RICH FEATURES ---
1. NCA...
2. Isolation Forests...
3. Autoencoder...
Training Transformer with Dual Loss...
Epoch 05 | Total Loss: 0.8707 | Triplet Part: 0.4513
Epoch 10 | Total Loss: 0.7739 | Triplet Part: 0.3650
Epoch 15 | Total Loss: 0.6234 | Triplet Part: 0.2227
Epoch 20 | Total Loss: 0.5653 | Triplet Part: 0.1673

--- PHASE E: TUNING INFERENCE ---
Searching best thresholds...

   👑 RESULTS (FT-Transformer + Triplet): 73.90% (MCC: 0.6211) 👑
              precision    recall  f1-score   support

         dos       0.96      0.75      0.84      7458
      normal       0.65      0.97      0.78      9711
       probe       0.71      0.60      0.65      2421
         r2l       0.76      0.07      0.12      2887
         u2r       0.00      0.00      0.00        67

    accuracy                           0.74     22544
   macro avg       0.62      0.48      0.48     22544